# Image Colorization — ResNet18 U-Net + Loss Percettiva
### Versione ottimizzata macOS Apple Silicon (M1/M2/M3): MPS · No AMP · PyTorch LAB conversion · DataLoader senza spawn
---
**Architettura:** ResNet18 (encoder preaddestrato) + U-Net decoder con skip connections  
**Loss:** SmoothL1 + Perceptual Loss (VGG16, conv on-device) + SSIM Loss  
**Training:** 3 fasi — decoder only → fine-tuning encoder → LR cosine restart  
**Fix Apple Silicon:** `num_workers=0` · `pin_memory=False` · `GradScaler` disabilitato · `_to_rgb` su MPS senza numpy roundtrip  


## 0. Download Dataset COCO 2017

In [ ]:
import os, subprocess, shutil

dest = "/kaggle/working/dataset"  # ← cambia se non sei su Kaggle
os.makedirs(dest, exist_ok=True)

subprocess.run([
    "kaggle", "datasets", "download",
    "-d", "awsaf49/coco-2017-dataset",
    "-p", dest, "--unzip"
], check=True)
print("Download completato.")

## 1. Import e configurazione

In [2]:
import os, math, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights, VGG16_Weights
from skimage.color import rgb2lab, lab2rgb
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

warnings.filterwarnings('ignore')

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"MPS:  {torch.backends.mps.is_available()}")

PyTorch: 2.11.0
CUDA: False
MPS:  True


## 2. Configurazione centralizzata (CONFIG)

In [3]:
# ─────────────────────────────────────────────────────────────
#  Tutte le impostazioni in un unico dizionario — cambia solo qui
# ─────────────────────────────────────────────────────────────
CONFIG = {
    # Paths
    "train_dir":       "//Users/micheleukmar/Documents/visual studio/progetto deep learning/dataset/coco2017/train2017",
    "val_dir":         "//Users/micheleukmar/Documents/visual studio/progetto deep learning/dataset/coco2017/val2017",
    "checkpoint_dir":  "//Users/micheleukmar/Documents/visual studio/progetto deep learning/checkpoints/checkTest",

    # Dataset
    "image_size":      256,
    "max_train":       60000,   # None = usa tutto
    "max_val":         5000,
    "seed":            42,

    # DataLoader
    # M2 8GB  → batch_size 8-12   (safe)
    # M2 16GB → batch_size 16-32  (comodo)
    # M2 Pro/Max 32GB+ → batch_size 32-64
    "batch_size":      16,
    "num_workers":     0,        # DEVE essere 0 su macOS (spawn issue)
    "prefetch_factor": 2,        # ignorato se num_workers=0

    # Training fasi
    "epochs_fase1":    12,
    "epochs_fase2":    15,
    "epochs_fase3":    10,

    # Optimizer
    "lr_decoder":      1e-3,
    "lr_encoder":      5e-6,

    # Loss weights
    "w_smooth":        1.0,
    "w_perceptual":    0.1,
    "w_ssim":          0.2,

    # Early stopping
    "patience":        5,

    # Mixed precision (AMP)
    # MPS (Apple Silicon) supporta autocast ma NON GradScaler fp16 → use_amp=False
    # Su CUDA Ampere+ → metti True per speedup reale
    "use_amp":         False,

    # torch.compile (PyTorch 2.x) — su MPS è sperimentale, lascia False
    "use_compile":     False,

    # Gradient clipping
    "grad_clip":       1.0,
}

os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps"  if torch.backends.mps.is_available()
    else "cpu"
)

# AMP: attivo solo su CUDA
CONFIG["use_amp"] = CONFIG["use_amp"] and device.type == "cuda"

# pin_memory: utile solo su CUDA (host→GPU DMA); su MPS/CPU è inutile o dannoso
CONFIG["pin_memory"] = device.type == "cuda"

print(f"Device:     {device}")
print(f"AMP:        {CONFIG['use_amp']}")
print(f"pin_memory: {CONFIG['pin_memory']}")
print(f"batch_size: {CONFIG['batch_size']}")


Device:     mps
AMP:        False
pin_memory: False
batch_size: 16


## 3. Dataset con augmentation avanzata

In [4]:
class ColorizationDataset(Dataset):
    """
    Restituisce (L, ab) nello spazio colore LAB normalizzato.
    L in [-1, 1], ab in [-1, 1].
    Augmentation: flip, crop, jitter — solo in modalità train.
    """
    def __init__(self, image_paths, size=256, is_train=True):
        self.paths    = image_paths
        self.size     = size
        self.is_train = is_train

        if is_train:
            self.aug = transforms.Compose([
                transforms.Resize(int(size * 1.12), Image.BICUBIC),
                transforms.RandomCrop(size),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(p=0.1),
                transforms.ColorJitter(
                    brightness=0.3, contrast=0.3,
                    saturation=0.5, hue=0.05
                ),
                transforms.RandomApply(
                    [transforms.GaussianBlur(3, sigma=(0.1, 2.0))], p=0.2
                ),
            ])
        else:
            self.aug = transforms.Resize((size, size), Image.BICUBIC)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        for attempt in range(5):
            try:
                img    = Image.open(self.paths[(idx + attempt) % len(self.paths)]).convert('RGB')
                img    = self.aug(img)
                img_np = np.array(img, dtype=np.float32) / 255.0
                lab    = rgb2lab(img_np).astype(np.float32)

                L  = torch.from_numpy((lab[:, :, 0] / 50.0) - 1.0).unsqueeze(0)
                ab = torch.from_numpy(lab[:, :, 1:] / 128.0).permute(2, 0, 1)
                return L, ab
            except Exception:
                continue
        # fallback: tensore nero
        return torch.zeros(1, self.size, self.size), torch.zeros(2, self.size, self.size)

## 4. DataLoader ottimizzato

In [5]:
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

def load_paths(folder, max_n=None):
    exts = ('.jpg', '.jpeg', '.png')
    paths = [os.path.join(folder, f) for f in os.listdir(folder)
             if f.lower().endswith(exts)]
    if max_n and len(paths) > max_n:
        paths = list(np.random.choice(paths, max_n, replace=False))
    return paths

train_imgs = load_paths(CONFIG["train_dir"], CONFIG["max_train"])
val_imgs   = load_paths(CONFIG["val_dir"],   CONFIG["max_val"])

train_set = ColorizationDataset(train_imgs, CONFIG["image_size"], is_train=True)
val_set   = ColorizationDataset(val_imgs,   CONFIG["image_size"], is_train=False)

# prefetch_factor è valido solo se num_workers > 0
loader_kwargs = dict(
    batch_size         = CONFIG["batch_size"],
    num_workers        = CONFIG["num_workers"],
    pin_memory         = CONFIG["pin_memory"],
    persistent_workers = (CONFIG["num_workers"] > 0),
)
if CONFIG["num_workers"] > 0:
    loader_kwargs["prefetch_factor"] = CONFIG["prefetch_factor"]

train_loader = DataLoader(train_set, shuffle=True,  **loader_kwargs)
val_loader   = DataLoader(val_set,   shuffle=False, **loader_kwargs)

print(f"Train: {len(train_set)} immagini | Val: {len(val_set)} immagini")
print(f"Batch size: {CONFIG['batch_size']} | Batch/epoch: {len(train_loader)}")


Train: 60000 immagini | Val: 5000 immagini
Batch size: 16 | Batch/epoch: 3750


## 5. Architettura — ResNet18 U-Net con Attention Gate

In [6]:
# ── Attention Gate ─────────────────────────────────────────────
class AttentionGate(nn.Module):
    """Seleziona le feature skip più rilevanti prima della concatenazione."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, 1, bias=False),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, 1, bias=False),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )

    def forward(self, g, x):
        # g = gate (decoder), x = skip (encoder)
        # Riportiamo g alla stessa risoluzione di x se necessario
        if g.shape[-2:] != x.shape[-2:]:
            g = F.interpolate(g, size=x.shape[-2:], mode='bilinear', align_corners=False)
        att = F.relu(self.W_g(g) + self.W_x(x), inplace=True)
        att = self.psi(att)
        return x * att


# ── Decoder Block con residual connection ──────────────────────
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.attn = AttentionGate(out_ch, skip_ch, skip_ch // 2)
        merged_ch  = out_ch + skip_ch
        self.conv = nn.Sequential(
            nn.Conv2d(merged_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.residual = nn.Conv2d(merged_ch, out_ch, 1, bias=False)
        self.act = nn.GELU()

    def forward(self, x, skip):
        x    = self.up(x)
        skip = self.attn(x, skip)       # attention-gated skip
        cat  = torch.cat([x, skip], 1)
        return self.act(self.conv(cat) + self.residual(cat))


# ── Modello principale ─────────────────────────────────────────
class ColorizerResNet(nn.Module):
    """
    ResNet18 encoder (pretrained) + U-Net decoder con Attention Gate.
    Input:  L   (B, 1, H, W)  normalizzato [-1,1]
    Output: ab  (B, 2, H, W)  normalizzato [-1,1]  (via Tanh)
    """
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)

        # Encoder layers
        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # →  64 ch, /2
        self.enc2 = nn.Sequential(resnet.maxpool, resnet.layer1)          # →  64 ch, /4
        self.enc3 = resnet.layer2                                          # → 128 ch, /8
        self.enc4 = resnet.layer3                                          # → 256 ch, /16

        # Bottleneck leggero
        self.bottleneck = nn.Sequential(
            nn.Conv2d(256, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.GELU(),
            nn.Dropout2d(0.2),
        )

        # Decoder
        self.dec4 = DecoderBlock(256, 128, 128)   # + skip enc3
        self.dec3 = DecoderBlock(128,  64,  64)   # + skip enc2
        self.dec2 = DecoderBlock( 64,  64,  64)   # + skip enc1
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
        )

        self.out = nn.Conv2d(32, 2, 1)
        self.tanh = nn.Tanh()

        self._init_decoder_weights()

    def _init_decoder_weights(self):
        for m in [self.dec4, self.dec3, self.dec2, self.dec1, self.out, self.bottleneck]:
            for layer in (m.modules() if hasattr(m, 'modules') else [m]):
                if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d)):
                    nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')

    def forward(self, L):
        x  = L.repeat(1, 3, 1, 1)    # (B,1,H,W) → (B,3,H,W)
        e1 = self.enc1(x)             # (B, 64, H/2,  W/2)
        e2 = self.enc2(e1)            # (B, 64, H/4,  W/4)
        e3 = self.enc3(e2)            # (B,128, H/8,  W/8)
        e4 = self.enc4(e3)            # (B,256, H/16, W/16)
        b  = self.bottleneck(e4)
        d4 = self.dec4(b,  e3)        # (B,128, H/8,  W/8)
        d3 = self.dec3(d4, e2)        # (B, 64, H/4,  W/4)
        d2 = self.dec2(d3, e1)        # (B, 64, H/2,  W/2)
        d1 = self.dec1(d2)            # (B, 32, H,    W)
        return self.tanh(self.out(d1))

    def congela_encoder(self):
        for part in [self.enc1, self.enc2, self.enc3, self.enc4]:
            for p in part.parameters(): p.requires_grad_(False)

    def scongela_encoder(self):
        for p in self.parameters(): p.requires_grad_(True)


# Verifica shape
with torch.no_grad():
    m = ColorizerResNet()
    out = m(torch.zeros(2, 1, 256, 256))
    print(f"Output shape: {out.shape}  ✓")  # atteso: (2, 2, 256, 256)
    params_tot = sum(p.numel() for p in m.parameters()) / 1e6
    params_dec = sum(p.numel() for p in list(m.dec4.parameters()) +
                     list(m.dec3.parameters()) + list(m.dec2.parameters()) +
                     list(m.dec1.parameters()) + list(m.out.parameters())) / 1e6
    print(f"Parametri totali: {params_tot:.2f}M  |  Decoder: {params_dec:.2f}M")

Output shape: torch.Size([2, 2, 256, 256])  ✓
Parametri totali: 4.31M  |  Decoder: 0.94M


## 6. Loss composita: SmoothL1 + Perceptual (VGG) + SSIM

In [7]:
# ── SSIM Loss ─────────────────────────────────────────────────
class SSIMLoss(nn.Module):
    """Structural Similarity loss (differenziabile, single-scale)."""
    def __init__(self, window_size=11):
        super().__init__()
        self.window_size = window_size
        self.C1, self.C2 = 0.01**2, 0.03**2
        sigma = 1.5
        coords = torch.arange(window_size, dtype=torch.float32) - window_size // 2
        g = torch.exp(-(coords**2) / (2 * sigma**2))
        g /= g.sum()
        kernel = g.outer(g).unsqueeze(0).unsqueeze(0)   # (1,1,W,W)
        self.register_buffer('kernel', kernel)

    def _conv(self, x, c):
        k = self.kernel.expand(c, 1, -1, -1)
        return F.conv2d(x, k, padding=self.window_size//2, groups=c)

    def forward(self, pred, target):
        c = pred.shape[1]
        mu1 = self._conv(pred, c); mu2 = self._conv(target, c)
        s1  = self._conv(pred*pred, c)     - mu1*mu1
        s2  = self._conv(target*target, c) - mu2*mu2
        s12 = self._conv(pred*target, c)   - mu1*mu2
        ssim = ((2*mu1*mu2 + self.C1)*(2*s12 + self.C2)) / \
               ((mu1**2 + mu2**2 + self.C1)*(s1 + s2 + self.C2))
        return 1 - ssim.mean()


# ── Perceptual Loss (VGG16 relu2_2 + relu3_3) ─────────────────
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
        self.slice1 = vgg[:9]    # relu2_2
        self.slice2 = vgg[9:16]  # relu3_3
        for p in self.parameters():
            p.requires_grad_(False)
        self.register_buffer('mean', torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer('std',  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
        # Matrice XYZ→sRGB lineare (illuminante D65)
        self.register_buffer('xyz2rgb_mat', torch.tensor([
            [ 3.2404542, -1.5371385, -0.4985314],
            [-0.9692660,  1.8760108,  0.0415560],
            [ 0.0556434, -0.2040259,  1.0572252],
        ]))

    def _lab_to_rgb(self, L_norm, ab_norm):
        """
        Conversione LAB→RGB interamente su device (no numpy, no CPU roundtrip).
        Input:  L_norm  (B,1,H,W) in [-1,1]
                ab_norm (B,2,H,W) in [-1,1]
        Output: rgb     (B,3,H,W) in [0,1], normalizzato ImageNet
        """
        # Denormalizza
        L  = (L_norm  + 1.0) * 50.0          # [0, 100]
        a  = ab_norm[:, 0:1] * 128.0          # [-128, 127]
        b  = ab_norm[:, 1:2] * 128.0

        # LAB → XYZ (D65)
        fy = (L + 16.0) / 116.0
        fx = a / 500.0 + fy
        fz = fy - b / 200.0

        eps = 6.0 / 29.0   # ~0.2069
        kap = eps ** 2      # per la parte lineare

        def f_inv(t):
            return torch.where(t > eps, t.pow(3), (t - 4.0 / 29.0) * 3.0 * kap)

        X = 0.95047 * f_inv(fx)
        Y = 1.00000 * f_inv(fy)
        Z = 1.08883 * f_inv(fz)

        # XYZ → RGB lineare
        xyz = torch.cat([X, Y, Z], dim=1)                    # (B,3,H,W)
        B_sz, _, H, W = xyz.shape
        M = self.xyz2rgb_mat.to(xyz.dtype)                    # (3,3)
        rgb = torch.einsum('oi,bihw->bohw', M, xyz)          # (B,3,H,W)

        # Gamma sRGB
        rgb = rgb.clamp(0.0, 1.0)
        rgb = torch.where(
            rgb <= 0.0031308,
            12.92 * rgb,
            1.055 * rgb.pow(1.0 / 2.4) - 0.055
        ).clamp(0.0, 1.0)

        # Normalizzazione ImageNet
        return (rgb - self.mean) / self.std

    def forward(self, pred_ab, target_ab, L):
        # Usa float32 esplicito (VGG non va con fp16)
        pred_rgb   = self._lab_to_rgb(L.float(),      pred_ab.float())
        target_rgb = self._lab_to_rgb(L.float(), target_ab.float())
        f1_p = self.slice1(pred_rgb);   f1_t = self.slice1(target_rgb)
        f2_p = self.slice2(f1_p);       f2_t = self.slice2(f1_t)
        return F.l1_loss(f1_p, f1_t) + F.l1_loss(f2_p, f2_t)


# ── Loss composita ─────────────────────────────────────────────
class CompositeLoss(nn.Module):
    def __init__(self, w_smooth=1.0, w_perceptual=0.1, w_ssim=0.2):
        super().__init__()
        self.smooth     = nn.SmoothL1Loss()
        self.perceptual = PerceptualLoss()
        self.ssim       = SSIMLoss()
        self.w_smooth   = w_smooth
        self.w_perc     = w_perceptual
        self.w_ssim     = w_ssim

    def forward(self, pred, target, L):
        l_smooth = self.smooth(pred, target)
        l_ssim   = self.ssim(pred, target)
        l_perc   = self.perceptual(pred, target, L)
        total    = (self.w_smooth * l_smooth
                    + self.w_ssim  * l_ssim
                    + self.w_perc  * l_perc)
        return total, {"smooth": l_smooth.item(),
                       "ssim":   l_ssim.item(),
                       "perc":   l_perc.item()}

print("Loss composita pronta ✓")


Loss composita pronta ✓


## 7. Loop di training ottimizzato (AMP + GradScaler + gradient clipping)

In [8]:
def train_epoch(model, loader, criterion, optimizer, scaler, device, grad_clip=1.0):
    model.train()
    total_loss = 0.0
    for L, ab in tqdm(loader, desc='  Train', leave=False):
        L, ab = L.to(device, non_blocking=True), ab.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)  # più efficiente di zero_grad()

        with torch.autocast(device_type=device.type, enabled=CONFIG["use_amp"]):
            pred = model(L)
            loss = F.l1_loss(pred, ab)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    for L, ab in tqdm(loader, desc='  Val  ', leave=False):
        L, ab = L.to(device, non_blocking=True), ab.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=CONFIG["use_amp"]):
            pred = model(L)
            loss = F.l1_loss(pred, ab)
        total_loss += loss.item()
    return total_loss / len(loader)


# ── Checkpoint ────────────────────────────────────────────────
def save_ckpt(state, path):
    torch.save(state, path)
    print(f"  ✓ Checkpoint salvato → {os.path.basename(path)}")


def load_ckpt(path, model, optimizer=None, scheduler=None):
    if not os.path.exists(path):
        return 0, [], []
    print(f"Carico checkpoint: {path}")
    ckpt = torch.load(path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'])
        if optimizer and 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if scheduler and 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        return ckpt.get('epoch', 0), ckpt.get('train_losses', []), ckpt.get('val_losses', [])
    else:
        model.load_state_dict(ckpt)
        return 0, [], []


# ── EarlyStopping ─────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience=5, delta=1e-4):
        self.patience = patience
        self.delta    = delta
        self.best     = float('inf')
        self.counter  = 0
        self.stop     = False

    def __call__(self, val_loss):
        if val_loss < self.best - self.delta:
            self.best    = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
        return self.stop

print("Funzioni training pronte ✓")

Funzioni training pronte ✓


## 8. Inizializzazione modello e loss

In [9]:
model     = ColorizerResNet().to(device)
criterion = CompositeLoss(
    w_smooth=CONFIG["w_smooth"],
    w_perceptual=CONFIG["w_perceptual"],
    w_ssim=CONFIG["w_ssim"]
).to(device)

# GradScaler: usa torch.amp (API moderna), abilitato solo su CUDA
# Su MPS/CPU use_amp è già False → scaler è un no-op
scaler = torch.amp.GradScaler('cuda', enabled=CONFIG["use_amp"])

# torch.compile — accelerazione GPU Ampere+; su MPS è sperimentale → False
if CONFIG["use_compile"] and hasattr(torch, 'compile'):
    model = torch.compile(model)
    print("torch.compile attivato")

ckpt_f1 = os.path.join(CONFIG["checkpoint_dir"], 'colorizer_fase1_best.pth')
ckpt_f2 = os.path.join(CONFIG["checkpoint_dir"], 'colorizer_fase2_best.pth')
ckpt_f3 = os.path.join(CONFIG["checkpoint_dir"], 'colorizer_finale.pth')

print(f"Modello su {device} | Parametri: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")


Modello su mps | Parametri: 4.31M


## 9. FASE 1 — Solo Decoder (Encoder congelato)

In [11]:
model.congela_encoder()

optimizer_f1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["lr_decoder"], weight_decay=1e-4
)
# CosineAnnealingWarmRestarts: LR scende a coseno, poi riparte — evita local minima
scheduler_f1 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_f1, T_0=4, T_mult=2, eta_min=1e-6
)
early_stop_f1 = EarlyStopping(patience=CONFIG["patience"])

start_ep, train_l_f1, val_l_f1 = load_ckpt(ckpt_f1, model, optimizer_f1)
best_val_f1 = min(val_l_f1) if val_l_f1 else float('inf')

if start_ep >= CONFIG["epochs_fase1"]:
    print("Fase 1 già completata.")
else:
    print(f"\n── FASE 1: Decoder only ({CONFIG['epochs_fase1']} epoche) ──")
    for ep in range(start_ep + 1, CONFIG["epochs_fase1"] + 1):
        t_loss = train_epoch(model, train_loader, criterion, optimizer_f1, scaler, device, CONFIG["grad_clip"])
        v_loss = val_epoch(model, val_loader, criterion, device)
        scheduler_f1.step(ep)  # CosineAnnealingWarmRestarts vuole l'indice globale

        train_l_f1.append(t_loss)
        val_l_f1.append(v_loss)

        lr_now = optimizer_f1.param_groups[0]['lr']
        print(f"Ep {ep:02d}/{CONFIG['epochs_fase1']} | Train {t_loss:.4f} | Val {v_loss:.4f} | LR {lr_now:.2e}")

        if v_loss < best_val_f1:
            best_val_f1 = v_loss
            save_ckpt({'epoch': ep, 'model_state_dict': model.state_dict(),
                       'optimizer_state_dict': optimizer_f1.state_dict(),
                       'train_losses': train_l_f1, 'val_losses': val_l_f1}, ckpt_f1)

        if early_stop_f1(v_loss):
            print(f"  Early stopping a epoca {ep}")
            break

    print(f"Best val loss Fase 1: {best_val_f1:.4f}")

Carico checkpoint: //Users/micheleukmar/Documents/visual studio/progetto deep learning/checkpoints/checkTest/colorizer_fase1_best.pth

── FASE 1: Decoder only (12 epoche) ──


Ep 09/12 | Train 0.0650 | Val 0.0635 | LR 3.09e-04
  ✓ Checkpoint salvato → colorizer_fase1_best.pth


Ep 10/12 | Train 0.0648 | Val 0.0633 | LR 1.47e-04
  ✓ Checkpoint salvato → colorizer_fase1_best.pth


Ep 11/12 | Train 0.0645 | Val 0.0632 | LR 3.90e-05
  ✓ Checkpoint salvato → colorizer_fase1_best.pth


Ep 12/12 | Train 0.0646 | Val 0.0632 | LR 1.00e-03
  ✓ Checkpoint salvato → colorizer_fase1_best.pth
Best val loss Fase 1: 0.0632


## 10. FASE 2 — Fine-tuning completo con LR differenziato

In [13]:
model.scongela_encoder()

optimizer_f2 = torch.optim.AdamW([
    {'params': model.enc1.parameters(), 'lr': CONFIG["lr_encoder"]},
    {'params': model.enc2.parameters(), 'lr': CONFIG["lr_encoder"]},
    {'params': model.enc3.parameters(), 'lr': CONFIG["lr_encoder"]},
    {'params': model.enc4.parameters(), 'lr': CONFIG["lr_encoder"]},
    {'params': model.bottleneck.parameters(), 'lr': CONFIG["lr_decoder"] / 5},
    {'params': model.dec4.parameters(), 'lr': CONFIG["lr_decoder"] / 5},
    {'params': model.dec3.parameters(), 'lr': CONFIG["lr_decoder"] / 5},
    {'params': model.dec2.parameters(), 'lr': CONFIG["lr_decoder"] / 5},
    {'params': model.dec1.parameters(), 'lr': CONFIG["lr_decoder"] / 5},
    {'params': model.out.parameters(),  'lr': CONFIG["lr_decoder"] / 5},
], weight_decay=1e-4)

scheduler_f2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_f2, mode='min', factor=0.5, patience=3, min_lr=1e-7
)
early_stop_f2 = EarlyStopping(patience=CONFIG["patience"])

# Carica fase2 se esiste, altrimenti parte da fase1
if os.path.exists(ckpt_f2):
    start_ep2, train_l_f2, val_l_f2 = load_ckpt(ckpt_f2, model, optimizer_f2, scheduler_f2)
else:
    print("Carico pesi Fase 1...")
    load_ckpt(ckpt_f1, model)
    start_ep2, train_l_f2, val_l_f2 = 0, [], []

best_val_f2 = min(val_l_f2) if val_l_f2 else float('inf')

if start_ep2 >= CONFIG["epochs_fase2"]:
    print("Fase 2 già completata.")
else:
    print(f"\n── FASE 2: Fine-tuning ({CONFIG['epochs_fase2']} epoche) ──")
    for ep in range(start_ep2 + 1, CONFIG["epochs_fase2"] + 1):
        t_loss = train_epoch(model, train_loader, criterion, optimizer_f2, scaler, device, CONFIG["grad_clip"])
        v_loss = val_epoch(model, val_loader, criterion, device)
        scheduler_f2.step(v_loss)

        train_l_f2.append(t_loss)
        val_l_f2.append(v_loss)

        lr_now = optimizer_f2.param_groups[0]['lr']
        print(f"Ep {ep:02d}/{CONFIG['epochs_fase2']} | Train {t_loss:.4f} | Val {v_loss:.4f} | LR {lr_now:.2e}")

        if v_loss < best_val_f2:
            best_val_f2 = v_loss
            save_ckpt({'epoch': ep, 'model_state_dict': model.state_dict(),
                       'optimizer_state_dict': optimizer_f2.state_dict(),
                       'scheduler_state_dict': scheduler_f2.state_dict(),
                       'train_losses': train_l_f2, 'val_losses': val_l_f2}, ckpt_f2)

        if early_stop_f2(v_loss):
            print(f"  Early stopping a epoca {ep}")
            break

    print(f"Best val loss Fase 2: {best_val_f2:.4f}")

Carico checkpoint: //Users/micheleukmar/Documents/visual studio/progetto deep learning/checkpoints/checkTest/colorizer_fase2_best.pth

── FASE 2: Fine-tuning (15 epoche) ──


KeyboardInterrupt: 

## 11. FASE 3 — Ciclo finale

In [ ]:
scheduler_f3 = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_f2, T_0=5, eta_min=1e-8
)
early_stop_f3 = EarlyStopping(patience=CONFIG["patience"] + 2)

if os.path.exists(ckpt_f3):
    start_ep3, train_l_f3, val_l_f3 = load_ckpt(ckpt_f3, model, optimizer_f2, scheduler_f3)
else:
    load_ckpt(ckpt_f2, model)
    start_ep3, train_l_f3, val_l_f3 = 0, [], []

best_val_f3 = min(val_l_f3) if val_l_f3 else float('inf')

print(f"\n── FASE 3: Ciclo finale ({CONFIG['epochs_fase3']} epoche) ──")
for ep in range(start_ep3 + 1, start_ep3 + CONFIG["epochs_fase3"] + 1):
    t_loss = train_epoch(model, train_loader, criterion, optimizer_f2, scaler, device, CONFIG["grad_clip"])
    v_loss = val_epoch(model, val_loader, criterion, device)
    scheduler_f3.step(ep)

    train_l_f3.append(t_loss)
    val_l_f3.append(v_loss)
    print(f"Ep {ep:02d} | Train {t_loss:.4f} | Val {v_loss:.4f}")

    if v_loss < best_val_f3:
        best_val_f3 = v_loss
        save_ckpt({'epoch': ep, 'model_state_dict': model.state_dict(),
                   'optimizer_state_dict': optimizer_f2.state_dict(),
                   'scheduler_state_dict': scheduler_f3.state_dict(),
                   'train_losses': train_l_f3, 'val_losses': val_l_f3}, ckpt_f3)

    if early_stop_f3(v_loss):
        print(f"  Early stopping a epoca {ep}")
        break

print(f"Best val loss Fase 3: {best_val_f3:.4f}")

Carico checkpoint: //Users/micheleukmar/Documents/visual studio/progetto deep learning/checkpoints/checkTest/colorizer_fase2_best.pth

── FASE 3: Cicli cosine (10 epoche) ──


Ep 01 | Train 0.0639 | Val 0.0625
  ✓ Checkpoint salvato → colorizer_finale.pth


KeyboardInterrupt: 

## 12. Grafici training

In [14]:
def plot_losses(losses_dict, title="Training Losses"):
    n = len(losses_dict)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 4))
    if n == 1: axes = [axes]
    colors = {'Train': 'steelblue', 'Val': 'orangered'}
    for ax, (phase, (tl, vl)) in zip(axes, losses_dict.items()):
        ax.plot(tl, label='Train', color=colors['Train'], lw=2)
        ax.plot(vl, label='Val',   color=colors['Val'],   lw=2, ls='--')
        ax.set_title(phase, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.legend(); ax.grid(alpha=0.3)
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_losses({
    "Fase 1 — Decoder only":    (train_l_f1, val_l_f1),
    "Fase 2 — Fine-tuning":     (train_l_f2, val_l_f2),
    "Fase 3 — Cosine restart":  (train_l_f3, val_l_f3),
}, title="Image Colorization — Loss per Fase")

NameError: name 'train_l_f3' is not defined

## 13. Utility di visualizzazione

In [ ]:
def lab_to_rgb(L_tensor, ab_tensor, sat_boost=1.0):
    """Converte tensori L e ab normalizzati in immagine RGB numpy."""
    L_np  = (L_tensor.squeeze().cpu().numpy() + 1.0) * 50.0
    ab_np = ab_tensor.permute(1, 2, 0).cpu().numpy() * 128.0 * sat_boost
    ab_np = np.clip(ab_np, -128, 127)
    lab   = np.stack([L_np, ab_np[:,:,0], ab_np[:,:,1]], axis=2)
    return np.clip(lab2rgb(lab), 0, 1)


@torch.no_grad()
def mostra_risultati(model, dataset, n=6, sat_boost=1.2):
    """Mostra n campioni: B&N | Predetto | Originale."""
    model.eval()
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    for ax, title in zip(axes[0], ['Input (B&N)', 'Predetto', 'Originale']):
        ax.set_title(title, fontweight='bold', fontsize=12)
    for row, idx in enumerate(indices):
        L, ab_true = dataset[int(idx)]
        ab_pred = model(L.unsqueeze(0).to(device)).squeeze(0).cpu()
        axes[row, 0].imshow(L.squeeze().numpy(), cmap='gray')
        axes[row, 1].imshow(lab_to_rgb(L, ab_pred, sat_boost))
        axes[row, 2].imshow(lab_to_rgb(L, ab_true))
        for ax in axes[row]: ax.axis('off')
    plt.suptitle('Risultati Colorizzazione', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('risultati_colorization.png', dpi=150, bbox_inches='tight')
    plt.show()


# Carica il miglior modello e mostra risultati
load_ckpt(ckpt_f3, model)
mostra_risultati(model, val_set, n=8, sat_boost=1.2)

## 14. Colora immagine esterna

In [ ]:
@torch.no_grad()
def colora_immagine(model, image_path, size=256, sat_boost=1.2):
    model.eval()
    img    = Image.open(image_path).convert('RGB').resize((size, size), Image.BICUBIC)
    img_np = np.array(img, dtype=np.float32) / 255.0
    lab    = rgb2lab(img_np).astype(np.float32)
    L      = torch.from_numpy((lab[:,:,0] / 50.0) - 1.0).unsqueeze(0).unsqueeze(0)

    with torch.autocast(device_type=device.type, enabled=CONFIG["use_amp"]):
        ab_pred = model(L.to(device)).squeeze(0).cpu().float()

    img_out = lab_to_rgb(L.squeeze(0), ab_pred, sat_boost)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(L.squeeze().numpy(), cmap='gray'); axes[0].set_title('Input B&N')
    axes[1].imshow(img_out);                          axes[1].set_title('Colorizzato')
    axes[2].imshow(img_np);                           axes[2].set_title('Originale')
    for ax in axes: ax.axis('off')
    plt.suptitle(os.path.basename(image_path))
    plt.tight_layout(); plt.show()

    out_path = 'output_colorizzato.jpg'
    Image.fromarray((img_out * 255).astype(np.uint8)).save(out_path)
    print(f"Salvato → {out_path}")
    return img_out

# Test su immagine casuale del validation set
colora_immagine(model, val_imgs[0])
# colora_immagine(model, 'mia_foto.jpg')  # ← tua immagine

## 15. Salvataggio finale e riepilogo

In [ ]:
final_path = 'colorizer_finale_v2.pth'
torch.save(model.state_dict(), final_path)
size_mb = os.path.getsize(final_path) / 1e6

print("=" * 60)
print("RIEPILOGO PROGETTO v2")
print("=" * 60)
print(f"Dataset:         COCO 2017")
print(f"Spazio colore:   LAB  (L input, ab target)")
print(f"Architettura:    ResNet18 U-Net + Attention Gate")
print(f"Loss:            SmoothL1 + Perceptual (VGG16) + SSIM")
print(f"Optimizer:       AdamW (weight_decay=1e-4)")
print(f"Scheduler F1:    CosineAnnealingWarmRestarts")
print(f"Scheduler F2:    ReduceLROnPlateau")
print(f"Scheduler F3:    CosineAnnealingWarmRestarts")
print(f"AMP:             {CONFIG['use_amp']}")
print(f"EarlyStopping:   patience={CONFIG['patience']}")
print(f"Train images:    {len(train_set)}")
print(f"Val images:      {len(val_set)}")
print(f"Modello salvato: {final_path}  ({size_mb:.1f} MB)")
print("=" * 60)